In [8]:
pip install pyserial


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()
for port in ports:
    print(f"Device found: {port.device} - {port.description}")

Device found: /dev/cu.debug-console - n/a
Device found: /dev/cu.Bluetooth-Incoming-Port - n/a
Device found: /dev/cu.usbmodemSDA6C2E1E721 - OpenSDA Hardware


In [ ]:
import serial
import time

# --- Serial Configuration ---
SERIAL_PORT = '/dev/cu.usbmodemSDA6C2E1E721'
BAUD_RATE = 115200

# --- Game State Variables ---
score = 0
combo = 0
max_combo = 0

def process_hit(hit_type):
    """Process hit type and update score and combo."""
    global score, combo, max_combo
    
    if hit_type == "P":
        score += 100
        combo += 1
        print(f"🕺 PERFECT! (+100) | 👑 Combo: {combo} | 💯 Total Score: {score}")
    elif hit_type == "G":
        score += 50
        combo += 1
        print(f"👍 GOOD!    (+50)  | 👑 Combo: {combo} | 💯 Total Score: {score}")
    elif hit_type == "M":
        combo = 0
        print(f"❌ MISS!    (+0)   | 💔 Combo broken! | 💯 Total Score: {score}")
    
    # Update max combo
    if combo > max_combo:
        max_combo = combo

print("Config loaded, connecting to hardware...")

配置加载完毕，准备连接硬件...


In [ ]:
try:
    with serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0.5) as ser:
        print(f"✅ Connected to {SERIAL_PORT}")
        time.sleep(1.5)
        
        ser.reset_input_buffer() 
        ser.reset_output_buffer()
        
        print("🚀 Sending START command...")
        ser.write(b'S') 
        print("-" * 40)
        print("🎮 Game started! Waiting for player actions...\n")

        while True:
            if ser.in_waiting > 0:
                try:
                    raw_line = ser.readline().decode('utf-8').strip()
                except UnicodeDecodeError:
                    continue

                if not raw_line:
                    continue
                    
                # Parse the line for hit results
                if raw_line.startswith("HIT:"):
                    # Extract the judgment result after ':'
                    hit_result = raw_line.split(":")[1]
                    process_hit(hit_result)
                else:
                    # Print debug information sent by the development board
                    print(f"🔧 [Board Log] {raw_line}")
                    
except KeyboardInterrupt:
    print("\n" + "=" * 40)
    print("🛑 Game Over (Manually Stopped)")
    print(f"🏆 Final Score: {score}")
    print(f"🔥 Max Combo: {max_combo}")
    print("=" * 40)
    
    try:
        with serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0.5) as ser:
            ser.write(b'X')
    except Exception:
        pass

✅ 成功连接到 /dev/cu.usbmodemSDA6C2E1E721
🚀 发送 START 指令...
----------------------------------------
🎮 游戏开始！等待玩家动作...


🛑 游戏结束 (手动停止)
🏆 最终得分: 5850
🔥 最大连击: 2
